# T3 - Backend Orchestration API

**Guardian Eye task:** T3

Run this notebook top-to-bottom on the target virtual machine. Configuration is centralized near the top, and heavy model work only occurs in explicitly marked execution cells.


## 1. API schemas and imports

Defines frontend-ready response contracts while keeping large frame tensors outside JSON.


In [ ]:
from __future__ import annotations
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any, Callable
from uuid import uuid4
import importlib
import os
import shutil

from fastapi import FastAPI, File, HTTPException, UploadFile
from pydantic import BaseModel, Field

class AnalyzeResponse(BaseModel):
    verdict: str
    confidence: float = Field(ge=0, le=1)
    threshold: float
    evidence_packet: dict[str, Any]
    explanation: str
    guardrail_status: str
    incident_id: str
    recap_frame_paths: list[str]
    limitations_note: str

class IncidentSearchRequest(BaseModel):
    query: str
    top_k: int = Field(default=5, ge=1, le=50)

class IncidentRecord(BaseModel):
    incident_id: str
    timestamp: str
    verdict: str
    confidence: float
    packet_summary: str
    narrative: str
    frames_ref: list[str] = []


## 2. Configurable integration adapters

Each owner can expose their implementation as a normal Python callable. Environment variables select the concrete functions without changing orchestration code.


In [ ]:
def resolve(spec: str) -> Callable[..., Any]:
    module_name, attr = spec.split(":", maxsplit=1)
    return getattr(importlib.import_module(module_name), attr)

ADAPTER_SPECS = {
    "classifier": os.getenv("GE_CLASSIFIER", "models.classifier.entrypoint:analyze_clip_core"),
    "packet_builder": os.getenv("GE_PACKET_BUILDER", "rag.evidence_packet:build_evidence_packet"),
    "reference_retriever": os.getenv("GE_REFERENCE_RETRIEVER", "rag.reference_store:retrieve_reference"),
    "narrator": os.getenv("GE_NARRATOR", "models.narrator.explain:generate_clip_explanation"),
    "incident_writer": os.getenv("GE_INCIDENT_WRITER", "rag.incident_memory:insert_incident"),
    "incident_search": os.getenv("GE_INCIDENT_SEARCH", "rag.incident_memory:search_incidents"),
    "incident_get": os.getenv("GE_INCIDENT_GET", "rag.incident_memory:get_incident"),
    "recap_builder": os.getenv("GE_RECAP_BUILDER", "models.narrator.recap:create_visual_recap"),
}

def load_adapters() -> dict[str, Callable[..., Any]]:
    return {name: resolve(spec) for name, spec in ADAPTER_SPECS.items()}


## 3. Orchestrator

Runs the required order: classifier, packet, reference retrieval, narrator, recap, incident write. The classifier result remains authoritative.


In [ ]:
class GuardianEyeOrchestrator:
    def __init__(self, adapters: dict[str, Callable[..., Any]]):
        self.a = adapters

    def analyze(self, video_path: str | Path) -> AnalyzeResponse:
        core = self.a["classifier"](video_path)
        packet = self.a["packet_builder"](core.telemetry, core.gate, core.gqs)
        refs = self.a["reference_retriever"](packet["packet_summary"], top_k=4)
        narrative = self.a["narrator"](
            frames=core.frames_vit,
            packet=packet,
            refs=refs,
            verdict=core.verdict,
            confidence=core.confidence,
        )
        if narrative.get("verdict", core.verdict) != core.verdict:
            raise RuntimeError("Narrator guardrail failure: classifier verdict was contradicted")
        recap_paths = self.a["recap_builder"](core.frames_vit, packet)
        incident = self.a["incident_writer"](
            {
                **core.json_metadata(),
                **packet,
                "narrative": narrative["narrative"],
                "frames_ref": recap_paths,
            }
        )
        return AnalyzeResponse(
            verdict=core.verdict,
            confidence=core.confidence,
            threshold=core.threshold,
            evidence_packet=packet,
            explanation=narrative["narrative"],
            guardrail_status=narrative.get("guardrail_status", "passed"),
            incident_id=incident["incident_id"],
            recap_frame_paths=recap_paths,
            limitations_note=narrative.get(
                "limitations_note",
                "Timing and person-level claims are geometry-derived; the classifier is clip-level.",
            ),
        )


## 4. FastAPI application and endpoints

Creates `/analyze`, `/incidents/search`, and `/incidents/{id}`. Uploaded videos are removed after processing.


In [ ]:
UPLOAD_DIR = Path(os.getenv("GE_UPLOAD_DIR", "data/incoming"))
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.adapters = load_adapters()
    app.state.orchestrator = GuardianEyeOrchestrator(app.state.adapters)
    yield

app = FastAPI(title="Guardian Eye API", version="1.0.0", lifespan=lifespan)

@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok", "mode": "offline"}

@app.post("/analyze", response_model=AnalyzeResponse)
def analyze(file: UploadFile = File(...)) -> AnalyzeResponse:
    suffix = Path(file.filename or "clip.mp4").suffix or ".mp4"
    temp_path = UPLOAD_DIR / f"{uuid4().hex}{suffix}"
    try:
        with temp_path.open("wb") as target:
            shutil.copyfileobj(file.file, target)
        return app.state.orchestrator.analyze(temp_path)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc
    finally:
        temp_path.unlink(missing_ok=True)

@app.post("/incidents/search", response_model=list[IncidentRecord])
def search_incidents(request: IncidentSearchRequest):
    return app.state.adapters["incident_search"](request.query, top_k=request.top_k)

@app.get("/incidents/{incident_id}", response_model=IncidentRecord)
def get_incident(incident_id: str):
    record = app.state.adapters["incident_get"](incident_id)
    if record is None:
        raise HTTPException(status_code=404, detail="Incident not found")
    return record


## 5. Target-VM launch command

Run this only after all owner modules are importable. The app stays fully local.


In [ ]:
# Save/export this notebook's API code into backend/api/main.py, then launch:
# uvicorn backend.api.main:app --host 0.0.0.0 --port 8000
#
# Quick checks:
# curl http://127.0.0.1:8000/health
# curl -F "file=@data/incoming/example.mp4" http://127.0.0.1:8000/analyze
